In [1]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

IMAGE_SIZE = 256
BATCH_SIZE = 16
CHANNELS = 3
EPOCHS = 25
SEED = 12

DATA_DIR = "data"

os.makedirs("models", exist_ok=True)

In [3]:
# ============================================================
# LOAD DATASET
# ============================================================

dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    shuffle=True,
    seed=SEED,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = dataset.class_names
NUM_CLASSES = len(class_names)

print(
    f"Classes ({NUM_CLASSES}):",
    class_names
)


Found 9842 files belonging to 4 classes.
Classes (4): ['Damage-Pest', 'Healthy', 'Leaf-blight', 'unknown']


In [4]:
# ============================================================
# SAVE CLASS NAMES
# ============================================================

with open(
    "models/class_names.json",
    "w"
) as f:

    json.dump(
        class_names,
        f,
        indent=4
    )


In [5]:
# ============================================================
# DATASET SPLIT
# ============================================================

dataset_batches = (
    tf.data.experimental
    .cardinality(dataset)
    .numpy()
)

print(
    "Total batches:",
    dataset_batches
)

train_batches = int(
    0.80 * dataset_batches
)

val_batches = int(
    0.10 * dataset_batches
)

test_batches = (
    dataset_batches -
    train_batches -
    val_batches
)

print("Train batches:", train_batches)
print("Validation batches:", val_batches)
print("Test batches:", test_batches)


# Shuffle deterministically
dataset = dataset.shuffle(
    buffer_size=dataset_batches,
    seed=SEED,
    reshuffle_each_iteration=False
)


# Split
train_ds = dataset.take(
    train_batches
)

remaining_ds = dataset.skip(
    train_batches
)

val_ds = remaining_ds.take(
    val_batches
)

test_ds = remaining_ds.skip(
    val_batches
)

Total batches: 616
Train batches: 492
Validation batches: 61
Test batches: 63


In [6]:
# ============================================================
# PREFETCH
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(
    AUTOTUNE
)

val_ds = val_ds.prefetch(
    AUTOTUNE
)

test_ds = test_ds.prefetch(
    AUTOTUNE
)


print("\nDataset preparation complete.")


Dataset preparation complete.


In [7]:
# ============================================================
# DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal"
        ),

        layers.RandomRotation(
            0.05
        ),

        layers.RandomZoom(
            0.10
        ),
    ],
    name="data_augmentation"
)

print("Data augmentation created.")

Data augmentation created.


In [8]:
# ============================================================
# MOBILENETV2 BASE MODEL
# ============================================================

base_model = MobileNetV2(
    input_shape=(
        IMAGE_SIZE,
        IMAGE_SIZE,
        CHANNELS
    ),
    include_top=False,
    weights="imagenet"
)

# Freeze the pretrained model initially
base_model.trainable = False

print("MobileNetV2 loaded.")

C:\Users\innov\AppData\Local\Temp\ipykernel_19512\280470993.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


MobileNetV2 loaded.


In [9]:
# ============================================================
# BUILD CLASSIFICATION MODEL
# ============================================================

model = models.Sequential(
    [
        layers.Input(
            shape=(
                IMAGE_SIZE,
                IMAGE_SIZE,
                CHANNELS
            )
        ),

        # ----------------------------------------------------
        # DATA AUGMENTATION
        # Active only during training
        # ----------------------------------------------------
        data_augmentation,

        # ----------------------------------------------------
        # MOBILENETV2 PREPROCESSING
        # Converts [0,255] -> [-1,1]
        # ----------------------------------------------------
        layers.Lambda(
            preprocess_input,
            name="mobilenetv2_preprocess"
        ),

        # ----------------------------------------------------
        # PRETRAINED MOBILENETV2
        # ----------------------------------------------------
        base_model,

        # ----------------------------------------------------
        # CLASSIFICATION HEAD
        # ----------------------------------------------------
        layers.GlobalAveragePooling2D(),

        layers.BatchNormalization(),

        layers.Dropout(
            0.30
        ),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.BatchNormalization(),

        layers.Dropout(
            0.20
        ),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.Dropout(
            0.20
        ),

        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=
                tf.keras.regularizers.l2(
                    1e-4
                )
        ),

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------
        layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            name="classifier"
        )
    ],
    name="r4i_ginger_classifier"
)

In [10]:
# ============================================================
# MODEL SUMMARY
# ============================================================

model.summary()

Model: "r4i_ginger_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_preprocess (Lambda) │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,461,124 (9.39 MB)

 Trainable params: 200,324 (782.52 KB)

 Non-trainable params: 2,260,800 (8.62 MB)

In [11]:
# ============================================================
# COMPILE — STAGE 1
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False
    ),

    metrics=[
        "accuracy"
    ]
)

In [12]:
# ============================================================
# CALLBACKS
# ============================================================

checkpoint = ModelCheckpoint(
    "best_r4i_ginger.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

In [13]:
# ============================================================
# TRAIN — STAGE 1
# ============================================================

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

Epoch 1/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step - accuracy: 0.6884 - loss: 0.7476
Epoch 1: val_accuracy improved from None to 0.85041, saving model to best_r4i_ginger.keras
492/492 ━━━━━━━━━━━━━━━━━━━━ 162s 306ms/step - accuracy: 0.7564 - loss: 0.6045 - val_accuracy: 0.8504 - val_loss: 0.3641 - learning_rate: 0.0010
Epoch 2/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step - accuracy: 0.8142 - loss: 0.4686
Epoch 2: val_accuracy improved from 0.85041 to 0.87398, saving model to best_r4i_ginger.keras
492/492 ━━━━━━━━━━━━━━━━━━━━ 150s 289ms/step - accuracy: 0.8245 - loss: 0.4473 - val_accuracy: 0.8740 - val_loss: 0.3412 - learning_rate: 0.0010
Epoch 3/10
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.8374 - loss: 0.4143
Epoch 3: val_accuracy improved from 0.87398 to 0.87807, saving model to best_r4i_ginger.keras
492/492 ━━━━━━━━━━━━━━━━━━━━ 148s 290ms/step - accuracy: 0.8353 - loss: 0.4208 - val_accuracy: 0.8781 - val_loss: 0.2944 - learning_rate: 0.0010
Epoch 4/10
492/492 ━━━

In [14]:
# ============================================================
# FINE-TUNING
# ============================================================

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [15]:
for layer in base_model.layers:

    if isinstance(
        layer,
        layers.BatchNormalization
    ):
        layer.trainable = False

In [16]:
model.summary()

Model: "r4i_ginger_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_preprocess (Lambda) │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,861,774 (10.92 MB)

 Trainable params: 1,711,044 (6.53 MB)

 Non-trainable params: 750,080 (2.86 MB)

 Optimizer params: 400,650 (1.53 MB)

In [17]:
# ============================================================
# COMPILE — STAGE 2
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=False
    ),

    metrics=[
        "accuracy"
    ]
)

In [18]:
# ============================================================
# TRAIN — STAGE 2
# ============================================================

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

Epoch 1/25
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.8865 - loss: 0.2944
Epoch 1: val_accuracy did not improve from 0.92111
492/492 ━━━━━━━━━━━━━━━━━━━━ 171s 320ms/step - accuracy: 0.8923 - loss: 0.2814 - val_accuracy: 0.9201 - val_loss: 0.1972 - learning_rate: 1.0000e-05
Epoch 2/25
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - accuracy: 0.8879 - loss: 0.2740
Epoch 2: val_accuracy improved from 0.92111 to 0.95082, saving model to best_r4i_ginger.keras
492/492 ━━━━━━━━━━━━━━━━━━━━ 215s 352ms/step - accuracy: 0.8951 - loss: 0.2595 - val_accuracy: 0.9508 - val_loss: 0.1465 - learning_rate: 1.0000e-05
Epoch 3/25
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step - accuracy: 0.9116 - loss: 0.2488
Epoch 3: val_accuracy did not improve from 0.95082
492/492 ━━━━━━━━━━━━━━━━━━━━ 168s 324ms/step - accuracy: 0.9076 - loss: 0.2549 - val_accuracy: 0.9129 - val_loss: 0.2029 - learning_rate: 1.0000e-05
Epoch 4/25
492/492 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.9108 - loss: 0.2405
Epoch

In [19]:
# ============================================================
# FINAL TEST
# ============================================================

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(
    "\nTest accuracy:",
    test_accuracy
)

print(
    "Test loss:",
    test_loss
)

63/63 ━━━━━━━━━━━━━━━━━━━━ 26s 254ms/step - accuracy: 0.9315 - loss: 0.1847

Test accuracy: 0.9315476417541504
Test loss: 0.18470007181167603


In [20]:
# ============================================================
# SINGLE IMAGE TEST
# ============================================================

def predict_image(
    model,
    image_path
):

    image = tf.keras.utils.load_img(
        image_path,
        target_size=(
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    )

    image_array = tf.keras.utils.img_to_array(
        image
    )

    image_array = tf.expand_dims(
        image_array,
        axis=0
    )

    predictions = model.predict(
        image_array,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(predictions)
    )

    print("\nPrediction:")
    print(
        class_names[predicted_index]
    )

    print("\nProbabilities:")

    for name, probability in zip(
        class_names,
        predictions
    ):

        print(
            f"{name}: "
            f"{probability * 100:.2f}%"
        )

    return predictions

In [25]:
predict_image(
    model,
    "test_data/unknown.jpg"
)


Prediction:
unknown

Probabilities:
Damage-Pest: 0.00%
Healthy: 0.00%
Leaf-blight: 0.00%
unknown: 100.00%


array([1.2607872e-07, 2.2555733e-07, 2.7213596e-06, 9.9999690e-01],
      dtype=float32)

In [26]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in test_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        predicted_labels
    )

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

              precision    recall  f1-score   support

 Damage-Pest       0.95      0.93      0.94       213
     Healthy       0.83      0.97      0.90       225
 Leaf-blight       0.94      0.84      0.89       321
     unknown       1.00      1.00      1.00       249

    accuracy                           0.93      1008
   macro avg       0.93      0.94      0.93      1008
weighted avg       0.93      0.93      0.93      1008



In [34]:
# ============================================================
# CREATE INFERENCE MODEL FROM THE TRAINED MODEL
# REMOVE ONLY DATA AUGMENTATION
# KEEP ALL TRAINED WEIGHTS
# ============================================================

inputs = tf.keras.Input(
    shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
    name="image"
)

x = inputs

for layer in model.layers:

    # Skip the random augmentation layer
    if layer.name == "data_augmentation":
        print("Skipping:", layer.name)
        continue

    print("Adding:", layer.name)

    x = layer(x, training=False)

inference_model = tf.keras.Model(
    inputs=inputs,
    outputs=x,
    name="r4i_ginger_inference"
)

print("\nInference model created.")
inference_model.summary()

Skipping: data_augmentation
Adding: mobilenetv2_preprocess
Adding: mobilenetv2_1.00_224
Adding: global_average_pooling2d
Adding: batch_normalization
Adding: dropout
Adding: dense
Adding: batch_normalization_1
Adding: dropout_1
Adding: dense_1
Adding: dropout_2
Adding: dense_2
Adding: classifier

Inference model created.


Model: "r4i_ginger_inference"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_preprocess (Lambda) │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,461,124 (9.39 MB)

 Trainable params: 1,711,044 (6.53 MB)

 Non-trainable params: 750,080 (2.86 MB)

In [38]:
# ============================================================
# TEST INFERENCE MODEL
# ============================================================

image_path = "test_data/unknown.jpg"

image = tf.keras.utils.load_img(
    image_path,
    target_size=(IMAGE_SIZE, IMAGE_SIZE)
)

image_array = tf.keras.utils.img_to_array(image)
image_array = tf.expand_dims(image_array, axis=0)

prediction = inference_model.predict(
    image_array,
    verbose=0
)[0]

predicted_index = int(np.argmax(prediction))

print("\nPrediction:")
print(class_names[predicted_index])

print("\nProbabilities:")

for name, probability in zip(class_names, prediction):
    print(f"{name}: {probability * 100:.2f}%")


Prediction:
unknown

Probabilities:
Damage-Pest: 0.00%
Healthy: 0.00%
Leaf-blight: 0.00%
unknown: 100.00%


In [39]:
# ============================================================
# EXPORT CORRECTED INFERENCE MODEL
# ============================================================

inference_saved_model_path = (
    "r4i_ginger_inference_saved_model"
)

inference_model.export(
    inference_saved_model_path
)

print(
    "Inference SavedModel exported to:",
    inference_saved_model_path
)

INFO:tensorflow:Assets written to: r4i_ginger_inference_saved_model\assets


INFO:tensorflow:Assets written to: r4i_ginger_inference_saved_model\assets


Saved artifact at 'r4i_ginger_inference_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2279139270624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139545808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139545280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139542992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139544224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139551792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139550736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139553200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139548976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139551968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2279139556544: TensorSpec

In [40]:
# ============================================================
# TFLITE CONVERSION
# ============================================================

converter = tf.lite.TFLiteConverter.from_saved_model(
    inference_saved_model_path
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_model = converter.convert()

with open(
    "r4i_ginger.tflite",
    "wb"
) as f:
    f.write(tflite_model)

print("TFLite model saved successfully.")

TFLite model saved successfully.


In [41]:
# ============================================================
# VERIFY TFLITE MODEL
# ============================================================

interpreter = tf.lite.Interpreter(
    model_path="r4i_ginger.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nINPUT")
print("Shape:", input_details[0]["shape"])
print("Dtype:", input_details[0]["dtype"])

print("\nOUTPUT")
print("Shape:", output_details[0]["shape"])
print("Dtype:", output_details[0]["dtype"])


INPUT
Shape: [  1 256 256   3]
Dtype: <class 'numpy.float32'>

OUTPUT
Shape: [1 4]
Dtype: <class 'numpy.float32'>


C:\Users\innov\ml_env\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [45]:
# ============================================================
# TEST TFLITE MODEL
# ============================================================

image_path = "test_data/leafblight.jpg"

# Load image
image = tf.keras.utils.load_img(
    image_path,
    target_size=(IMAGE_SIZE, IMAGE_SIZE)
)

image_array = tf.keras.utils.img_to_array(image)
image_array = np.expand_dims(
    image_array,
    axis=0
).astype(np.float32)

# Run TFLite
interpreter.set_tensor(
    input_details[0]["index"],
    image_array
)

interpreter.invoke()

tflite_prediction = interpreter.get_tensor(
    output_details[0]["index"]
)[0]

tflite_index = int(
    np.argmax(tflite_prediction)
)

print("\nTFLITE PREDICTION")
print("=================")

print(
    "Prediction:",
    class_names[tflite_index]
)

print("\nProbabilities:")

for name, probability in zip(
    class_names,
    tflite_prediction
):
    print(
        f"{name}: {probability * 100:.2f}%"
    )


TFLITE PREDICTION
Prediction: Leaf-blight

Probabilities:
Damage-Pest: 20.79%
Healthy: 2.43%
Leaf-blight: 76.77%
unknown: 0.01%
